# Create Canada Research Chairs Awards (incl. Canada 150 Research Chairs)

Creates Canada Research Chairs (CRC) awards from the program's own publications
at https://www.chairs-chaires.gc.ca (Tri-agency Institutional Programs Secretariat):

1. **Recipients lists** (`/media-medias/lists-listes/{year}/{month}-eng.aspx`), one per
   allocation announcement, April 2015 to Spring 2026 (21 lists, 3,851 chair awards):
   chairholder, university, chair title, agency, tier, chair type (New / Renewal /
   Advancement). One award per (chair, announcement); `start_date` = first day of the
   announcement month.
2. **Chairholders database** (`/chairholders-titulaires/index-eng.aspx`, 1,941 current
   chairs): current chairs not found in any recipients list (same person + tier) are
   kept as undated awards (214, mostly awarded before April 2015).
3. **Canada 150 Research Chairs** laureates (canada150.chairs-chaires.gc.ca, 20 listed of
   24 awarded; one-time 2017-18 program run by the same secretariat). Canada 150 has no
   OpenAlex funder record, so these are attributed to CRC with
   `funder_scheme = 'Canada 150 Research Chairs'` and key prefix `C150-` so they can be
   split out later (**needs Kyle's call**).

**4,085 awards** (local run 2026-09-30); 100% title/institution/PI name, 94% dated.

**Amounts** are the program's published per-tier values (About page): Tier 1 =
$200,000/yr x 7 = CAD 1,400,000; Tier 2 = $100,000/yr x 5 = CAD 500,000 (the
first-term Tier 2 $20k/yr stipend is not flagged in the lists and not added).
Canada 150: the laureate's published "$N per year for seven years" x 7.

**`funder_award_id` (§2.1.1):** CRC publishes no award numbers on its site; citing
works write free text (CRC-YYYY-NNNNN, 950-NNNNNN, "Tier1", ...). So the key is
synthetic with prefixes that cannot collide with any existing CRC id in
`openalex_awards_raw` (checked 2026-09-30: no existing id matches the shapes below;
the only look-alikes are numeric `C150-2017-000NN`):
- recipients-list rows `CRC-T{tier}-{YYYY}-{MM}-{family-given slug}`
- database-only rows `CRC-T{tier}-{family-given slug}`
- Canada 150 rows `C150-{family-given slug}`

These rows therefore do not collapse onto existing citation shells. The same chairs
also appear, under the administering agency's funder id, in the NSERC/SSHRC/CIHR
open-data ingests (funder_scheme 'Canada Research Chairs'); different funder_id, so
no id collision.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320320994`
- display_name: Canada Research Chairs
- ror_id / doi: from `openalex.funders.funders` (ror 0517h6h17, doi 10.13039/501100001804)

**Priority:** `480` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.crc_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/crc/crc_chairs.parquet`;

In [ ]:
%sql
SELECT source, COUNT(*) as n FROM openalex.awards.crc_raw GROUP BY source;

In [ ]:
%sql
DESCRIBE openalex.awards.crc_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.crc_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320320994 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320320994;

## Step 2: Create CRC Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.crc_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320320994  -- Canada Research Chairs
),

lead AS (
    -- Names were split in Python ("Last, First" from the source; runbook §2.4.1).
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    SELECT
        funder_award_id,
        named_struct(
            'given_name', NULLIF(TRIM(lead_given_name), ''),
            'family_name', NULLIF(TRIM(lead_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', TRY_TO_DATE(announcement_date, 'yyyy-MM-dd'),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(institution), ''),
                'country', 'CA',
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) AS lead_investigator
    FROM openalex.awards.crc_raw
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.chair_title), '') as display_name,
    NULLIF(TRIM(g.description), '') as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    g.currency as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'fellowship' as funding_type,
    CASE
        WHEN g.funder_scheme = 'Canada 150 Research Chairs' THEN g.funder_scheme
        WHEN NULLIF(TRIM(g.tier), '') IS NOT NULL
            THEN CONCAT('Canada Research Chairs, Tier ', TRIM(g.tier),
                        CASE WHEN NULLIF(TRIM(g.chair_type), '') IS NOT NULL THEN CONCAT(' (', TRIM(g.chair_type), ')') ELSE '' END)
        ELSE 'Canada Research Chairs'
    END as funder_scheme,
    'crc_chairholders' as provenance,
    TRY_TO_DATE(g.announcement_date, 'yyyy-MM-dd') as start_date,
    CAST(NULL AS DATE) as end_date,
    YEAR(TRY_TO_DATE(g.announcement_date, 'yyyy-MM-dd')) as start_year,
    CAST(NULL AS INT) as end_year,
    l.lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    array(l.lead_investigator) as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.crc_raw g
JOIN lead l ON l.funder_award_id = g.funder_award_id
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

In [ ]:
%sql
-- Shape check (§2.1.1): only the synthetic prefixes; unique ids.
SELECT
    assert_true(COUNT_IF(NOT funder_award_id RLIKE '^(CRC-T[12X]-([0-9]{4}-[0-9]{2}-)?[a-z0-9-]+|C150-[a-z][a-z0-9-]*)$') = 0,
                'funder_award_id outside the CRC-T / C150- synthetic key shapes') AS shape_ok,
    assert_true(COUNT(*) = COUNT(DISTINCT id), 'duplicate award ids') AS unique_ok
FROM openalex.awards.crc_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'crc_chairholders' AND priority = 480;

-- Insert into openalex_awards_raw with priority.
-- Priority 480: direct-from-funder ingest of the CRC program's own recipients lists,
-- chairholders database and Canada 150 laureates. Higher wins under the 2026-06-20 DESC dedup.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    480 as priority
FROM openalex.awards.crc_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_crc_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.crc_awards;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_cad
FROM openalex.awards.crc_awards GROUP BY 1 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt FROM openalex.awards.crc_awards GROUP BY 1 ORDER BY 1;

In [ ]:
%sql
-- Section 6.4a: one chairholder legitimately appears at New and Renewal (and Advancement),
-- so 2-3 rows per person are expected; more than ~4 would be a parsing bug.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.crc_awards GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    collect_set(currency) as currencies,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.crc_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'crc_chairholders'
GROUP BY provenance, priority;